In [ ]:
# ── Google Colab Setup (skip if running locally) ──────────────────────
# This cell clones the repo and installs packages on Colab.
# On a local machine it does nothing (the 'src' folder already exists).
import os, sys

if not os.path.exists('src'):  # running on Colab
    !git clone https://github.com/AbhilashAgarwalIITJ/Thesis.git /content/thesis
    os.chdir('/content/thesis')
    !pip install networkx matplotlib numpy pandas tabulate -q
    print('Colab setup complete. Working directory:', os.getcwd())
else:
    print('Local environment detected. No setup needed.')


# Demo: AI-Augmented Verilog Netlist Matching

**Thesis:** AI-Augmented Verilog Netlist Matching Using AIG and Graph Neural Networks  
**Pipeline:** Verilog â†’ AIG â†’ Graph â†’ Cones â†’ Fingerprint / WL Hash â†’ Match  

This notebook walks through the core pipeline on a single example â€” the 4-bit adder and its carry-inversion mutant â€” to demonstrate the central result: WL hashing detects a false positive that structural fingerprinting misses.

## Step 1 â€” Load an AIG and build the graph

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from parse_aiger import parse_aiger_file, aiger_to_networkx

parsed = parse_aiger_file("aig_output/adder_4bit.aig")
G, po_nodes = aiger_to_networkx(parsed)

print(f"Design:  adder_4bit")
print(f"Nodes:   {G.number_of_nodes()}")
print(f"Edges:   {G.number_of_edges()}")
print(f"PO:      {len(po_nodes)}")
print(f"PI:      {sum(1 for n in G.nodes if G.nodes[n].get('type')=='PI')}")
print(f"AND:     {sum(1 for n in G.nodes if G.nodes[n].get('type')=='AND')}")

Design:  adder_4bit
Nodes:   51
Edges:   77
PO:      5
PI:      9
AND:     36


## Step 2 â€” Extract output cones

In [2]:
from cone_extract import extract_all_cones

cones = extract_all_cones(G, po_nodes)

print(f"{'Cone':<10} {'Nodes':>6} {'Edges':>6} {'AND':>5} {'PI':>4} {'Depth':>6}")
print("-" * 42)
for po, data in sorted(cones.items(), key=lambda x: x[1]['stats']['po_name']):
    s = data['stats']
    print(f"{s['po_name']:<10} {s['num_nodes']:>6} {s['num_edges']:>6} {s['num_and']:>5} {s['num_pi']:>4} {s['depth']:>6}")

Cone        Nodes  Edges   AND   PI  Depth
------------------------------------------
po_0           10     13     6    3      5
po_1           18     25    12    5      7
po_2           26     37    18    7      9
po_3           34     49    24    9     11
po_4           34     49    24    9      9


## Step 3 â€” Compute WL hashes (k=3, semantic + inversion-aware)

In [3]:
from wl_hash import wl_hash_all_cones

hashes_orig = wl_hash_all_cones(cones, k=3, semantic=True)

print(f"{'Cone':<10} {'WL Hash':>18}")
print("-" * 30)
for po, data in sorted(hashes_orig.items(), key=lambda x: x[1]['stats']['po_name']):
    print(f"{data['stats']['po_name']:<10} {data['hash']:>18}")

Cone                  WL Hash
------------------------------
po_0         a5c607be097e6412
po_1         07b0fd388876e3a5
po_2         ce26cab8583bc197
po_3         deda6e31555136f0
po_4         d894b1866ad5d3d7


## Step 4 â€” Load the carry-inversion mutant and compare

**Mutation:** `~c[2]` â€” the carry bit into bit 3 is inverted.  
All gate counts and depths stay identical. Can the methods detect it?

In [4]:
# Load mutant
parsed_mut = parse_aiger_file("aig_output/adder_4bit_mut2.aig")
G_mut, po_mut = aiger_to_networkx(parsed_mut)
cones_mut = extract_all_cones(G_mut, po_mut)
hashes_mut = wl_hash_all_cones(cones_mut, k=3, semantic=True)

print(f"Mutant loaded: {G_mut.number_of_nodes()} nodes, {len(po_mut)} POs")

Mutant loaded: 51 nodes, 5 POs


## Step 5 â€” Central result: Fingerprint vs WL comparison

In [5]:
def fingerprint(stats):
    return (stats['num_nodes'], stats['num_edges'], stats['num_and'], stats['num_pi'], stats['depth'])

# Align cones by PO name
orig_by_name = {d['stats']['po_name']: (po, d) for po, d in cones.items()}
mut_by_name  = {d['stats']['po_name']: (po, d) for po, d in cones_mut.items()}
hash_orig_by_name = {hashes_orig[po]['stats']['po_name']: hashes_orig[po] for po in hashes_orig}
hash_mut_by_name  = {hashes_mut[po]['stats']['po_name']:  hashes_mut[po]  for po in hashes_mut}

print(f"{'Cone':<8} {'Nodes A':>8} {'Nodes B':>8} {'FP Match':>9} {'WL Match':>9} {'Verdict'}")
print("-" * 62)

fp_match_count = 0
wl_match_count = 0

for name in sorted(orig_by_name.keys()):
    _, d_a = orig_by_name[name]
    _, d_b = mut_by_name[name]
    fp_a = fingerprint(d_a['stats'])
    fp_b = fingerprint(d_b['stats'])
    wl_a = hash_orig_by_name[name]['hash']
    wl_b = hash_mut_by_name[name]['hash']

    fp_eq = fp_a == fp_b
    wl_eq = wl_a == wl_b
    fp_match_count += fp_eq
    wl_match_count += wl_eq

    fp_str = "Yes" if fp_eq else "No"
    wl_str = "Yes" if wl_eq else "No"

    if fp_eq and not wl_eq:
        verdict = "<-- FALSE POSITIVE"
    elif fp_eq and wl_eq:
        verdict = "Both agree"
    else:
        verdict = "Both reject"

    print(f"{name:<8} {d_a['stats']['num_nodes']:>8} {d_b['stats']['num_nodes']:>8}"
          f"  {fp_str:>6}  {wl_str:>6}   {verdict}")

total = len(orig_by_name)
print(f"\nFingerprint match: {fp_match_count}/{total} ({100*fp_match_count//total}%)")
print(f"WL hash match:     {wl_match_count}/{total} ({100*wl_match_count//total}%)")
print(f"False positives caught by WL: {fp_match_count - wl_match_count}")

Cone      Nodes A  Nodes B  FP Match  WL Match Verdict
--------------------------------------------------------------
po_0           10       10     Yes     Yes   Both agree
po_1           18       18     Yes     Yes   Both agree
po_2           26       26     Yes     Yes   Both agree
po_3           34       34     Yes      No   <-- FALSE POSITIVE
po_4           34       34     Yes      No   <-- FALSE POSITIVE

Fingerprint match: 5/5 (100%)
WL hash match:     3/5 (60%)
False positives caught by WL: 2


## Step 6 â€” Visualise the AIG graph and one output cone

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Image

# Colour by node type
color_map = {'PI': '#4CAF50', 'AND': '#2196F3', 'PO': '#FF5722', 'CONST0': '#9E9E9E'}

# Pick the smallest cone for a clean visual
smallest_po = min(cones, key=lambda po: cones[po]['stats']['num_nodes'])
cone_data = cones[smallest_po]
cone_G = cone_data['subgraph']
cone_name = cone_data['stats']['po_name']

colors = [color_map.get(cone_G.nodes[n].get('type', ''), '#9E9E9E') for n in cone_G.nodes]

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
pos = nx.spring_layout(cone_G, seed=42)
nx.draw(cone_G, pos, ax=ax, with_labels=False, node_color=colors,
        node_size=120, arrows=True, edge_color='#666', width=0.8)

# Legend
for label, c in color_map.items():
    ax.scatter([], [], color=c, s=60, label=label)
ax.legend(loc='upper left', fontsize=8)
ax.set_title(f"Output cone: {cone_name}  ({cone_data['stats']['num_nodes']} nodes)", fontsize=11)
plt.tight_layout()
plt.savefig('results/plots/demo_cone.png', dpi=150)
plt.show()
print(f"Saved: results/plots/demo_cone.png")

Saved: results/plots/demo_cone.png


C:\Users\abhilash\AppData\Local\Temp\ipykernel_21208\4216026014.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

| Metric | Fingerprint | WL Hash (k=3) |
|--------|------------|----------------|
| Match rate (carry-inv) | 100% (wrong) | 60% (correct) |
| False positives | 2 cones | 0 cones |
| Detection | Misses inversion | Catches inversion |

**Central result:** The carry inversion preserves all aggregate metrics (node count, edge count, depth). Fingerprinting cannot distinguish the cones. WL hashing encodes the inversion pattern into neighbourhood signatures and correctly rejects `po_3` and `po_4`.